# Reproduce the manuscript computations

This notebook accompanies *A prime and powers of fixed bases* by
Cristian Cobeli, Nicolas Robles, and Alexandru Zaharescu.

Run the cells in order. They check the input hashes, run the complete exact
verifier, display the results, and regenerate the numerical tables.
The optional figures are regenerated when Matplotlib is installed.

The finite verification uses only Python's standard library.
The base-three computation verifies the candidate $118\,514\,161$
and the representation of every smaller admissible integer.

## Locate the repository

The notebook can be started from the repository root or from its
`notebooks` directory. The code finds the enclosing repository.

In [1]:
from pathlib import Path
from fractions import Fraction
import hashlib
import importlib.util
import json
import subprocess
import sys

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "verify_all.py").is_file()
     and (p / "data" / "source_manifest.json").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start this notebook from the cloned repository.")
print("Repository located.")
print("Python:", sys.version.split()[0])

Repository located.
Python: 3.12.14


## Check the input identities

The manifest identifies the original allocation witnesses and the
numerical tables extracted from v13. The full manuscript is not required
to run the verification.

In [2]:
manifest = json.loads((ROOT / "data" / "source_manifest.json").read_text())
for name, recorded in manifest["inputs"].items():
    data = (ROOT / name).read_bytes()
    digest = hashlib.sha256(data).hexdigest()
    if len(data) != recorded["bytes"] or digest != recorded["sha256"]:
        raise ValueError("Input does not match the manifest: " + name)
    print(name, ":", len(data), "bytes, hash verified")
print("Manuscript version:", manifest["manuscript"]["version"])
print("Manuscript SHA-256:", manifest["manuscript"]["sha256"])

binary_cover_1440_full.json : 7341 bytes, hash verified
ap_certificate.json : 2584 bytes, hash verified
data/tables_v13.json : 10250 bytes, hash verified
Manuscript version: v13
Manuscript SHA-256: af63bc8cb54d4ba3b2e93d6b068d223fce2c822a521640568a353e6a312b7df7


## Run every exact check

This cell invokes `verify_all.py` once. It verifies both reflected
allocation systems, all finite auxiliary order calculations, the
restricted and unrestricted ternary recurrences, the exact products and
coefficients, the introductory factorizations and congruences, the
balanced-triple density bound, the base-three search, and the checks
for two distinct bases.

The complete result files are written to `results/`. A failed check stops
the program. The displayed progress below omits the individual auxiliary
rows; the exact per-row results remain in the JSON output.

In [3]:
completed = subprocess.run(
    [sys.executable, str(ROOT / "verify_all.py")],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
if completed.returncode:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError("The exact verification failed.")
for line in completed.stdout.splitlines():
    if not line.startswith("TAIL "):
        print(line)

FINITE 2 guards 41 survivors 20548 minimum 3252320451/1562500000 decimal 2.08148508864 margin 6527/3125000000
FINITE 3 guards 30 survivors 2966 minimum 35740740681/15625000000 decimal 2.287407403584 margin 3153/7812500000
PRODUCT 2 1.1703148983591547 ZP 2.4359905655813083
PRODUCT 3 1.3319090110580183 ZP 3.0466179952571886
COEFFICIENT 2 {'lower': '1.642042484284197982', 'upper': '1.642042484284197983'}
COEFFICIENT 3 {'lower': '0.492349222099757667', 'upper': '0.492349222099757668'}
FULL_RECURSION 93 all equal
BASE3 118514161 153 complete factorizations passed
MOD600 39 empty; 1 sparse with J_5=2; 260 coprime
MOD60 class57: 599/601 empty; 17/19 witness 49 is coprime
MINIMALITY: see the base3 stage
BALANCED 626 primes; density at most 1623190827888387/6400000000000000
BASE3 minimality 118514161 39504717 smaller integers represented
COMMON SUMS 3 congruences; residue sets disjoint
ODD SUMS 4 rows of counts agree with the table
SUCCESS


## Allocation and auxiliary counts

Each prescribed prime is proved prime and checked to have its stated
order. The allocation checks cover every surviving exponent pair and
every divisor capacity. The auxiliary calculations use complete removal
of the small prime powers and a rigorous lower bound from the remaining
cofactor.

In [4]:
def read_result(name):
    return json.loads((ROOT / "results" / name).read_text())

finite = read_result("allocation_finite_exact.json")
tail = read_result("allocation_tail_exact.json")
coefficients = read_result("coefficient_checks.json")
recurrences = read_result("full_recursion_comparison.json")
intro = read_result("intro_checks.json")

print(f"{'Check':<35} {'Binary':>12} {'Ternary':>12}")
print("-" * 61)
items = [
    ("Prescribed prime entries",
     finite["2"]["guard_count"], finite["3"]["guard_count"]),
    ("Capacity inequalities",
     finite["2"]["capacities_checked"], finite["3"]["capacities_checked"]),
    ("Surviving exponent pairs",
     finite["2"]["surviving_pairs_full"], finite["3"]["surviving_pairs_full"]),
    ("Auxiliary order calculations",
     sum(len(row["orders"]) for row in tail["2"]["entries"]),
     sum(len(row["orders"]) for row in tail["3"]["entries"])),
    ("Finite covering counts",
     tail["2"]["finite_rows_checked"], tail["3"]["finite_rows_checked"]),
]
for label, binary, ternary in items:
    print(f"{label:<35} {binary:>12,} {ternary:>12,}")

for case in ("2", "3"):
    row = finite[case]
    print("\nCase", case)
    print("Minimum normalized coverage:", row["min_gamma_exact"])
    print("Target:", row["target_exact"])
    print("Positive margin:", row["target_margin_exact"])

print("\nTernary recurrences compared:", len(recurrences))
print("All restricted and unrestricted values agree:",
      all(row["restricted"] == row["full"] for row in recurrences))

Check                                     Binary      Ternary
-------------------------------------------------------------
Prescribed prime entries                      41           30
Capacity inequalities                         36           25
Surviving exponent pairs                  20,548        2,966
Auxiliary order calculations                 828        1,302
Finite covering counts                        92           93

Case 2
Minimum normalized coverage: 3252320451/1562500000
Target: 2081483/1000000
Positive margin: 6527/3125000000

Case 3
Minimum normalized coverage: 35740740681/15625000000
Target: 2287407/1000000
Positive margin: 3153/7812500000

Ternary recurrences compared: 93
All restricted and unrestricted values agree: True


## Products and coefficients

The verifier forms

$$
P_i=\prod_{r\in\mathcal R_i}
\left(1-\frac1r+\frac1{r-k_{i,r}}\right)
$$

as an exact rational number. The coefficients are

$$
C_2=\frac4{Z_2P_2},\qquad C_3=\frac3{2Z_3P_3}.
$$

The intervals below are proved by rational comparison. They are strict
enclosures, rather than rounding estimates.

In [5]:
for case in ("2", "3"):
    row = coefficients[case]
    print("Case", case)
    for field, label in (("P_interval", "P"),
                         ("ZP_interval", "Z*P"),
                         ("C_interval", "C")):
        interval = row[field]
        print(" ", interval["lower"], "<", label, "<", interval["upper"])
    exact = Fraction(row["C_exact"])
    target = Fraction(row["C_target_exact"])
    print("  Exact comparison with", target, ":", exact < target)
    margin = row["C_target_margin_interval"]
    print("  Positive margin lies between",
          margin["lower"], "and", margin["upper"])
    print()

Case 2
  1.170314898359154654 < P < 1.170314898359154655
  2.435990565581308307 < Z*P < 2.435990565581308308
  1.642042484284197982 < C < 1.642042484284197983
  Exact comparison with 1642043/1000000 : True
  Positive margin lies between 0.000000515715802017 and 0.000000515715802018

Case 3
  1.331909011058018356 < P < 1.331909011058018357
  3.046617995257188594 < Z*P < 3.046617995257188595
  0.492349222099757667 < C < 0.492349222099757668
  Exact comparison with 9847/20000 : True
  Positive margin lies between 0.000000777900242332 and 0.000000777900242333



## Introductory arithmetic

For $N=118\,514\,161$, all possible exponents satisfy $0\le a,b\le16$.
Ordering them gives 153 pairs. Each difference $N-3^a-3^b$ is completely
factored and has at least two distinct prime factors. This proves the
candidate's nonrepresentability. A separate stage shows that every odd
integer $n$ coprime to $3$ with $10<n<N$ is a prime plus two positive
powers of $3$.

The residue computation for $(599,601)$ gives 39 empty classes, one
sparse class, and 260 classes with a coprime prime residue modulo 600.
The density consequences use the analytic argument in the manuscript.

In [6]:
print("Base-three candidate:", intro["base3_candidate"])
print("Unordered exponent pairs:", intro["base3_candidate_pairs_checked"])
print("Minimum number of distinct prime factors in a difference:",
      min(len(row["factorization"]) for row in intro["base3_factorizations"]))
base3 = read_result("base3_minimality.json")
print("Smaller admissible integers, all represented:",
      base3["smaller_integers_count"])
print("Minimality checked:", base3["minimality_checked"])
print()
print("Modulo 600:")
print("  Empty classes:", intro["classes_empty"])
print("  Sparse classes:", intro["classes_sparse"])
print("  Classes with a coprime residue:",
      intro["classes_positive_density_subject_to_theorem"])
print("For (17,19) modulo 60, (exponent, exponent, prime residue):")
for witness in intro["17_19_coprime_witnesses"]:
    print(" ", witness)

Base-three candidate: 118514161
Unordered exponent pairs: 153
Minimum number of distinct prime factors in a difference: 2
Smaller admissible integers, all represented: 39504717
Minimality checked: True

Modulo 600:
  Empty classes: 39
  Sparse classes: 1
  Classes with a coprime residue: 260
For (17,19) modulo 60, (exponent, exponent, prime residue):
  [2, 1, 49]
  [2, 2, 7]
  [4, 1, 37]


## Regenerate the tables and figures

The table generator uses the exact inputs and result files. It requires
only the Python standard library. The optional figures use Matplotlib.
Install it with `python3 -m pip install -r requirements-figures.txt`
before running this cell if the figures need to be regenerated.

In [7]:
tables = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "make_tables.py")],
    cwd=ROOT, capture_output=True, text=True,
)
if tables.returncode:
    print(tables.stderr)
    raise RuntimeError("Table generation failed.")
print("CSV and LaTeX tables regenerated.")

if importlib.util.find_spec("matplotlib") is not None:
    figures = subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "make_plots.py")],
        cwd=ROOT, capture_output=True, text=True,
    )
    if figures.returncode:
        print(figures.stderr)
        raise RuntimeError("Figure generation failed.")
    print("PNG and SVG figures regenerated.")
else:
    print("Figures retained. Install requirements-figures.txt to regenerate them.")

CSV and LaTeX tables regenerated.
PNG and SVG figures regenerated.


### Auxiliary covering counts

The plot displays the certified values at prime indices through 499.

![Auxiliary covering counts](../plots/auxiliary_families.png)

### Partial rational products

Each plotted point is obtained from a rational partial product before
conversion to a plotting coordinate. The exact coefficient comparisons
are recorded above and in the result files.

![Partial rational products](../plots/partial_products.png)

The [methods](../docs/methods.md) explain the arithmetic in more detail.
The [versioning notes](../docs/versioning.md) describe the source identity
and how to record later manuscript revisions.